In [ ]:
import pandas as pd

df = pd.read_csv("Labels/TrainLabels.csv")
df.head()

In [ ]:
import os
import pandas as pd

# Load labels
df = pd.read_csv("Labels/TrainLabels.csv")

# Keep only ClipID and Engagement
df = df[["ClipID", "Engagement"]]

# Remove .avi from ClipID
df["ClipID"] = df["ClipID"].str.replace(".avi", "", regex=False)

image_paths = []
labels = []

train_path = "DataSet/Train"

for user in os.listdir(train_path):
    user_path = os.path.join(train_path, user)

    for clip in os.listdir(user_path):
        clip_path = os.path.join(user_path, clip)

        # Get engagement label
        label_row = df[df["ClipID"] == clip]

        if len(label_row) == 0:
            continue

        engagement_label = int(label_row["Engagement"].values[0])

        for img in os.listdir(clip_path):
            if img.endswith(".jpg"):
                image_paths.append(os.path.join(clip_path, img))
                labels.append(engagement_label)

print("Total images:", len(image_paths))

In [ ]:
import shutil
import os

source_root = "DataSet/Train"
dest_root = "data/extracted_frames"

os.makedirs(dest_root, exist_ok=True)

for root, dirs, files in os.walk(source_root):
    for file in files:
        if file.endswith(".jpg"):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dest_root, file)
            shutil.copy(src_path, dst_path)

print("Frames copied successfully.")

In [ ]:
import cv2
import os

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

input_folder = "data/extracted_frames"
output_folder = "data/cropped_faces"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = img[y:y+h, x:x+w]
        cv2.imwrite(os.path.join(output_folder, img_name), face)
        break  # take first detected face only

print("Face cropping done.")

In [ ]:
input_folder = "data/cropped_faces"
output_folder = "data/processed_faces"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    img = cv2.resize(img, (224, 224))
    cv2.imwrite(os.path.join(output_folder, img_name), img)

print("Preprocessing done.")

In [ ]:
import os

count = 0
for root, dirs, files in os.walk("data/processed_faces"):
    for file in files:
        if file.endswith(".jpg"):
            count += 1

print("Total processed images:", count)

In [ ]:
print("Extracted:", len(os.listdir("data/extracted_frames")))
print("Cropped:", len(os.listdir("data/cropped_faces")))
print("Processed:", len(os.listdir("data/processed_faces")))

In [ ]:
import pandas as pd

labels_path = "../Labels/TrainLabels.csv"   # adjust if name is different
df = pd.read_csv(labels_path)

df.head()

In [ ]:
import os

image_folder = "data/processed_faces"
images = os.listdir(image_folder)

data = []

for img in images:
    clip_id = img.split("_")[0] + ".avi"   # reconstruct original ClipID

    row = df[df["ClipID"] == clip_id]

    if not row.empty:
        engagement_label = row["Engagement"].values[0]   # example target
        data.append([os.path.join(image_folder, img), engagement_label])

final_df = pd.DataFrame(data, columns=["image_path", "label"])

print("Total matched images:", len(final_df))

In [ ]:
print("Total processed images:", len(images))

In [ ]:
print("Total label clips loaded:", len(df))
print("Sample ClipIDs:", df["ClipID"].head())

In [ ]:
final_df["label"].value_counts()

In [ ]:
final_df = final_df[final_df["label"] != 0]

print(final_df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.3,
    random_state=42,
    stratify=final_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

print("CSV files saved successfully.")

In [ ]:
train_df["label"].value_counts()